## Phase 1 — Load & Inspect Data

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('./data/house_prices.csv', engine='python')
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())


In [ ]:
df.head()


In [ ]:
df.info()


In [ ]:
# Missing values overview
df.isnull().sum()


## Phase 2 — Exploratory Data Analysis (EDA)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Price distribution
plt.figure(figsize=(10, 5))
sns.histplot(df['price_clean'].dropna(), bins=60, log_scale=True, color='#3498db')
plt.title('House Price Distribution (Log Scale)')
plt.xlabel('Price (INR)')
plt.tight_layout()
plt.show()


In [ ]:
# Price vs Carpet Area
plt.figure(figsize=(10, 5))
sns.scatterplot(data=df, x='carpet_area_sqft', y='price_clean', alpha=0.3, color='#e74c3c')
plt.title('House Price vs Carpet Area')
plt.xlabel('Carpet Area (sqft)')
plt.ylabel('Price (INR)')
plt.tight_layout()
plt.show()


In [ ]:
# Average price by top 15 locations
top_locations = (
    df.groupby('location')['price_clean']
    .mean()
    .sort_values(ascending=False)
    .head(15)
)

plt.figure(figsize=(12, 7))
sns.barplot(x=top_locations.values, y=top_locations.index, palette='viridis')
plt.title('Average House Price — Top 15 Locations')
plt.xlabel('Average Price (INR)')
plt.tight_layout()
plt.show()


In [ ]:
# Price by Furnishing Status
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='Furnishing', y='price_clean', palette='Set2')
plt.title('House Price by Furnishing Status')
plt.xlabel('Furnishing Status')
plt.ylabel('Price (INR)')
plt.tight_layout()
plt.show()


## Phase 3 — Cleaning & Feature Engineering

In [ ]:
# Load cleaned dataset
clean_df = pd.read_csv('./data/house_prices_cleaned.csv')
print('Cleaned dataset shape:', clean_df.shape)
print('Columns:', clean_df.columns.tolist())


In [ ]:
# Define Features (X) and Target (y)
numeric_features = [
    'carpet_area_sqft',
    'bathroom_num',
    'floor_num',
    'balcony_num',
    'parking_num'
]

categorical_features = [
    'location',
    'Furnishing',
    'Status',
    'Transaction',
    'Ownership',
    'facing'
]

all_features = numeric_features + categorical_features

X = clean_df[all_features].copy()
y = clean_df['target_price'].copy()

print('X shape:', X.shape)
print('y shape:', y.shape)


In [ ]:
# Log-transform target (heavily right-skewed)
y_log = np.log1p(y)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(y, bins=60, ax=axes[0], color='#e74c3c', edgecolor='white')
axes[0].set_title('Target Price — Original (skewed)')
axes[0].set_xlabel('Price (INR)')

sns.histplot(y_log, bins=60, ax=axes[1], color='#2ecc71', edgecolor='white')
axes[1].set_title('Target Price — Log Transformed (near-normal)')
axes[1].set_xlabel('log1p(Price)')

plt.tight_layout()
plt.show()

print('Skewness before:', round(y.skew(), 2))
print('Skewness after :', round(y_log.skew(), 2))


## Phase 4 — Preprocessing Pipeline & Train/Val/Test Split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 70 / 15 / 15 Split
X_train, X_temp, y_train, y_temp = train_test_split(X, y_log, test_size=0.30, random_state=42)
X_val,   X_test, y_val,   y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

total = len(X)
print(f'Train      : {len(X_train):>6}  ({len(X_train)/total*100:.1f}%)')
print(f'Validation : {len(X_val):>6}  ({len(X_val)/total*100:.1f}%)')
print(f'Test       : {len(X_test):>6}  ({len(X_test)/total*100:.1f}%)')


In [ ]:
# Numeric sub-pipeline: Median imputation + Standard scaling
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

# Categorical sub-pipeline: fill unknown + One-Hot Encode
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer,     numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

print('Preprocessing pipeline built successfully!')


## Phase 5 — Model Training & Selection

In [ ]:
import time
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

# Fit preprocessor on train only
X_train_proc = preprocessor.fit_transform(X_train)
X_val_proc   = preprocessor.transform(X_val)

n_features = X_train_proc.shape[1]

def evaluate_model(name, model, X_tr, y_tr, X_vl, y_vl):
    start = time.time()
    model.fit(X_tr, y_tr)
    elapsed = round(time.time() - start, 2)
    pred_val = model.predict(X_vl)
    return {
        'Model'    : name,
        'Val RMSE' : round(np.sqrt(mean_squared_error(y_vl, pred_val)), 4),
        'Val MAE'  : round(mean_absolute_error(y_vl, pred_val), 4),
        'Val R2'   : round(r2_score(y_vl, pred_val), 4),
        'Time (s)' : elapsed
    }, model

results = []
trained  = {}

# 1. Ridge Regression (Linear baseline)
r, m = evaluate_model('Ridge (Linear)', Ridge(alpha=1.0), X_train_proc, y_train, X_val_proc, y_val)
results.append(r); trained['Ridge (Linear)'] = m; print(f"> Ridge done — Val R2: {r['Val R2']}")

# 2. Random Forest
r, m = evaluate_model('Random Forest', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1), X_train_proc, y_train, X_val_proc, y_val)
results.append(r); trained['Random Forest'] = m; print(f"> Random Forest done — Val R2: {r['Val R2']}")

# 3. Monotonic XGBoost (our selected model)
constraints = [0] * n_features; constraints[0] = 1  # positive constraint on carpet_area_sqft
r, m = evaluate_model('XGBoost (Monotonic)', XGBRegressor(
    n_estimators=300, learning_rate=0.05, max_depth=7,
    subsample=0.8, colsample_bytree=0.8,
    monotone_constraints=tuple(constraints),
    random_state=42, n_jobs=-1, verbosity=0
), X_train_proc, y_train, X_val_proc, y_val)
results.append(r); trained['XGBoost (Monotonic)'] = m; print(f"> XGBoost (Monotonic) done — Val R2: {r['Val R2']}")

df_results = pd.DataFrame(results).set_index('Model')
print()
print(df_results.to_string())


## Phase 6 — Final Evaluation on Test Set

In [ ]:
# Use the best model: XGBoost (Monotonic)
best_model = trained['XGBoost (Monotonic)']

X_test_proc = preprocessor.transform(X_test)
pred_test_log = best_model.predict(X_test_proc)

test_rmse = np.sqrt(mean_squared_error(y_test, pred_test_log))
test_r2   = r2_score(y_test, pred_test_log)
test_mae_inr = mean_absolute_error(np.expm1(y_test), np.expm1(pred_test_log))

print('=' * 50)
print('  FINAL TEST SET EVALUATION')
print('=' * 50)
print(f'  Test RMSE (Log) : {test_rmse:.4f}')
print(f'  Test R2         : {test_r2:.4f}  ({test_r2*100:.2f}%)')
print(f'  Avg Error (INR) : {test_mae_inr:,.0f}  (~{test_mae_inr/100000:.2f} Lakhs)')
print('=' * 50)


In [ ]:
# Actual vs Predicted scatter plot
y_test_orig = np.expm1(y_test)
y_pred_orig = np.expm1(pred_test_log)

plt.figure(figsize=(8, 7))
plt.scatter(y_test_orig / 100_000, y_pred_orig / 100_000, alpha=0.3, color='#3498db', s=5)
plt.plot([0, y_test_orig.max()/100_000], [0, y_test_orig.max()/100_000], 'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Price (Lakhs)')
plt.ylabel('Predicted Price (Lakhs)')
plt.title(f'Actual vs Predicted — Test Set (R2 = {test_r2:.4f})')
plt.legend()
plt.tight_layout()
plt.show()


## Phase 7 — Export Model & Locations

In [ ]:
import joblib

# Build the final unified Pipeline (preprocessor + model together)
final_pipeline = Pipeline([
    ('prep', preprocessor),
    ('reg',  best_model)
])

# Fit on ALL data (train + val + test) for maximum coverage before export
final_pipeline.fit(X, y_log)

# Export unified model
joblib.dump(final_pipeline, './house_price.pkl')
print('Saved: house_price.pkl')


In [ ]:
# Export available locations list for frontend dropdown
locations = sorted(X['location'].dropna().unique().tolist())

import json
with open('./locations.json', 'w', encoding='utf-8') as f:
    json.dump(locations, f, indent=2)

print('Saved: locations.json')
print('Locations:', locations)


In [ ]:
# Quick sanity check on the exported pipeline
import numpy as np

test_input = pd.DataFrame([{
    'carpet_area_sqft': 1200.0,
    'bathroom_num': 2,
    'floor_num': 3,
    'balcony_num': 2,
    'parking_num': 1,
    'location': 'bangalore',
    'Furnishing': 'semi-furnished',
    'Status': 'ready to move',
    'Transaction': 'resale',
    'Ownership': 'freehold',
    'facing': 'east'
}])

pred = final_pipeline.predict(test_input)[0]
price = np.expm1(pred)
print(f'Sanity check — Bangalore 1200 sqft: INR {price:,.0f} (~{price/100000:.2f} Lakhs)')
print('Export successful!')
